In [1]:
import os 
import json,re
from enum import Enum
from openai import OpenAI
from pydantic import BaseModel

In [2]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPEN_AI_API")
try:
    client = OpenAI(api_key=api_key)
except ImportError:
    print("OpenAI library not found.")

In [5]:
def evaluate_with_llm_judge(query , context , generated_answer , model = "gpt-4o-mini"):
    """
    Uses an LLM to evaluate the quality of a RAG-generated answer.

    Args:
        query (str): The user's original query.
        context (str): The context retrieved by the RAG system.
        generated_answer (str): The answer generated by the RAG system.
        model (str): The OpenAI model to use as the judge.

    Returns:
        dict: A dictionary containing the judge's scores and reasoning.
              Returns None if the API call fails.
    """
    if not client:
        return {
            "error": "OpenAI client not initialized."
        }
        
    prompt = f"""
    You are an impartial judge evaluating the quality of an answer generated by a
    Retrieval-Augmented Generation (RAG) system.

    Your task is to evaluate the generated answer based on two criteria:
    1.  **Faithfulness**: Does the generated answer stay faithful to the provided context?
        It should not add information that is not present in the context or contradict it.
    2.  **Answer Relevance**: Is the generated answer relevant and helpful for the given query?

    You must provide a score from 1 to 5 for each criterion (1=Poor, 5=Excellent) and a brief
    explanation for your scores.

    **Query:**
    {query}

    **Retrieved Context:**
    {context}

    **Generated Answer:**
    {generated_answer}

    Please provide your evaluation *only* in a valid JSON format with the following keys:
    "faithfulness_score", "faithfulness_reasoning", "relevance_score", "relevance_reasoning".
    Your response MUST be a single JSON object and nothing else.
    """

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert evaluator of AI-generated text that responds only in valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            # The 'response_format' parameter is removed from here.
        )
        
        response_text = response.choices[0].message.content
        
        # Clean up the response to extract only the JSON part.
        # This handles cases where the model might wrap the JSON in ```json ... ```
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            evaluation = json.loads(json_str)
            return evaluation
        else:
            print("Error: Could not find a valid JSON object in the model's response.")
            print(f"Full response: {response_text}")
            return None

    except Exception as e:
        print(f"An error occurred during API call or JSON parsing: {e}")
        return None

In [6]:
evaluation_data = [
    {
        "query": "What is the primary function of mitochondria?",
        "context": "Mitochondria are organelles found in the cells of most eukaryotes. They are often referred to as the 'powerhouses' of the cell because they generate most of the cell's supply of adenosine triphosphate (ATP), used as a source of chemical energy.",
        "generated_answer": "The primary function of mitochondria is to act as the powerhouse of the cell, producing ATP."
    },
    {
        "query": "Who was the first person to walk on the moon?",
        "context": "The Apollo 11 mission was the first crewed mission to land on the Moon, on 20 July 1969. The first person to step onto the lunar surface was commander Neil Armstrong.",
        "generated_answer": "Neil Armstrong and Buzz Aldrin were the first people to walk on the moon."
    },
    {
        "query": "What is the boiling point of water?",
        "context": "At standard atmospheric pressure, water (H₂O) boils at 100° Celsius (212° Fahrenheit).",
        "generated_answer": "Water boils at 100 degrees."
    }
]

print("--- RAG Evaluation with LLM-as-a-Judge ---")
if not client:
    print("Cannot proceed without OpenAI client.")
else:
    for i, item in enumerate(evaluation_data):
        print(f"\n--- Evaluating Item {i+1} ---")
        print(f"Query: {item['query']}")
        print(f"Generated Answer: {item['generated_answer']}")

        evaluation_result = evaluate_with_llm_judge(
            item['query'],
            item['context'],
            item['generated_answer']
        )

        if evaluation_result:
            print("\n--- Judge's Evaluation ---")
            print(f"Faithfulness Score: {evaluation_result.get('faithfulness_score')}")
            print(f"Faithfulness Reasoning: {evaluation_result.get('faithfulness_reasoning')}")
            print(f"Relevance Score: {evaluation_result.get('relevance_score')}")
            print(f"Relevance Reasoning: {evaluation_result.get('relevance_reasoning')}")
            print("--------------------------")

--- RAG Evaluation with LLM-as-a-Judge ---

--- Evaluating Item 1 ---
Query: What is the primary function of mitochondria?
Generated Answer: The primary function of mitochondria is to act as the powerhouse of the cell, producing ATP.

--- Judge's Evaluation ---
Faithfulness Score: 5
Faithfulness Reasoning: The generated answer accurately reflects the information provided in the context, stating that mitochondria are the powerhouse of the cell and that they produce ATP, which is consistent with the retrieved context.
Relevance Score: 5
Relevance Reasoning: The generated answer directly addresses the query about the primary function of mitochondria, providing a clear and concise response that is both relevant and helpful.
--------------------------

--- Evaluating Item 2 ---
Query: Who was the first person to walk on the moon?
Generated Answer: Neil Armstrong and Buzz Aldrin were the first people to walk on the moon.

--- Judge's Evaluation ---
Faithfulness Score: 2
Faithfulness Reasonin